# Gaussian Naive Bayes — Irrigation Need Prediction
**Competition:** Kaggle Playground Series S6E4  
**Author:** Tyler Wolf Williams (@tylerwolfwilliams2)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SEED = 42

train = pd.read_csv('train.csv')
print('Shape:', train.shape)
print('\nClass distribution:')
print(train['Irrigation_Need'].value_counts())
print('\nClass proportions:')
print(train['Irrigation_Need'].value_counts(normalize=True).round(4))

Shape: (630000, 21)

Class distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Class proportions:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


## 1. Preprocessing

Adapted from the existing preprocessing pipeline in `Modeling_Bagging_RandomForest.ipynb`.

**Key difference from tree-based notebooks:** `StandardScaler` is applied to numerical features because GaussianNB is sensitive to feature magnitude (it computes per-feature Gaussian likelihoods), unlike tree models that only use rank order.

**Note on categorical features:** GaussianNB assumes all features are normally distributed. Categorical features encoded as integers (0–N) violate this assumption, but including them with `OrdinalEncoder` is an acceptable approximation and keeps the pipeline consistent with prior work. Only the Kaggle `train.csv` is used — no test.csv predictions are required for this assignment.

In [2]:
TARGET = 'Irrigation_Need'
DROP_COLS = ['id']

cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in DROP_COLS]

print('Categorical features:', cat_cols)
print('Numerical features:  ', num_cols)

feature_cols = num_cols + cat_cols
X = train[feature_cols].copy()
y_raw = train[TARGET].copy()

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw)
print('\nLabel encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# Stratified train/test split to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Encode categoricals (fit on train only)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols]  = oe.transform(X_test[cat_cols])

# Scale numericals (fit on train only)
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f'\nTrain: {X_train.shape} | Test: {X_test.shape}')

Categorical features: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical features:   ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']



Label encoding: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}



Train: (504000, 19) | Test: (126000, 19)


## 2. Train Gaussian Naive Bayes

In [3]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

print('GaussianNB trained.')
print(f'Classes (encoded): {gnb.classes_}  →  {le.classes_}')

GaussianNB trained.
Classes (encoded): [0 1 2]  →  ['High' 'Low' 'Medium']


## 3. Baseline Predictions (Default Rule)

The **default rule** assigns the class with the highest predicted probability — equivalent to `argmax` over `predict_proba`. This is what `gnb.predict()` returns.

In [4]:
# Predicted probabilities — columns match gnb.classes_ order: [High=0, Low=1, Medium=2]
proba = gnb.predict_proba(X_test)

# Default rule: argmax of probabilities
y_pred_baseline = gnb.predict(X_test)

print('=== Baseline Classification Report (Default Rule) ===')
print(classification_report(y_test, y_pred_baseline, target_names=le.classes_))

=== Baseline Classification Report (Default Rule) ===
              precision    recall  f1-score   support

        High       0.72      0.42      0.53      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.79      0.75      0.77     47815

    accuracy                           0.83    126000
   macro avg       0.79      0.69      0.73    126000
weighted avg       0.82      0.83      0.82    126000



## 4. Threshold Tuning — High Class

**Chosen class:** `High` — the rarest class (~3.3% of data) and the most consequential to miss. A field with high irrigation need that goes undetected risks crop failure.

**Chosen metric:** **Recall** for the High class. In this domain, a false negative (missed high-need field) is more costly than a false positive (unnecessary irrigation). Maximizing recall minimizes the worst outcome.

**Threshold logic:** The default argmax rule rarely assigns High because its class prior is low. By lowering the assignment threshold — assigning High whenever `P(High) >= threshold`, even if High isn't the top class — we can capture more true High cases at the cost of some precision.

In [5]:
from sklearn.metrics import recall_score, precision_score, f1_score

# High class is encoded as index 0 (alphabetical: High < Low < Medium)
high_idx = list(le.classes_).index('High')  # = 0
high_proba = proba[:, high_idx]

print('P(High) statistics on test set:')
print(f'  Mean:    {high_proba.mean():.4f}')
print(f'  Max:     {high_proba.max():.4f}')
for t in [0.05, 0.10, 0.15, 0.20]:
    pct = (high_proba >= t).mean() * 100
    print(f'  % >= {t:.2f}: {pct:.2f}%')

P(High) statistics on test set:
  Mean:    0.0330
  Max:     0.9390
  % >= 0.05: 11.08%
  % >= 0.10: 7.98%
  % >= 0.15: 6.48%
  % >= 0.20: 5.44%


In [6]:
# Scan thresholds to find the recall/precision tradeoff
print(f'  {"Threshold":<12} {"High Recall":<14} {"High Prec":<12} {"High F1":<10} {"Overall Acc"}')
print('-' * 65)

for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    y_pred_t = y_pred_baseline.copy()
    y_pred_t[high_proba >= t] = high_idx

    rec  = recall_score(y_test, y_pred_t, labels=[high_idx], average='macro')
    prec = precision_score(y_test, y_pred_t, labels=[high_idx], average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred_t, labels=[high_idx], average='macro', zero_division=0)
    acc  = (y_pred_t == y_test).mean()
    print(f'  {t:<12.2f} {rec:<14.4f} {prec:<12.4f} {f1:<10.4f} {acc:.4f}')

  Threshold    High Recall    High Prec    High F1    Overall Acc
-----------------------------------------------------------------
  0.05         0.9050         0.2724       0.4188     0.7769
  0.10         0.8701         0.3635       0.5127     0.8015
  0.15         0.8363         0.4303       0.5682     0.8124
  0.20         0.7953         0.4875       0.6045     0.8190
  0.25         0.7525         0.5482       0.6343     0.8240


  0.30         0.6885         0.5871       0.6337     0.8261


In [7]:
# Apply selected threshold
THRESHOLD = 0.10

y_pred_threshold = y_pred_baseline.copy()
y_pred_threshold[high_proba >= THRESHOLD] = high_idx

print(f'=== Threshold Predictions (High class threshold = {THRESHOLD}) ===')
print(classification_report(y_test, y_pred_threshold, target_names=le.classes_))

=== Threshold Predictions (High class threshold = 0.1) ===


              precision    recall  f1-score   support

        High       0.36      0.87      0.51      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.81      0.64      0.71     47815

    accuracy                           0.80    126000
   macro avg       0.68      0.80      0.70    126000
weighted avg       0.82      0.80      0.80    126000



## 5. Discussion

### Class Selected: High

The **High** class was selected because it is the rarest class (3.3% of all samples) and the most operationally important. In an irrigation context, failing to identify a high-need field — a false negative — can result in crop failure or permanent soil damage. An unnecessary irrigation event (false positive) wastes water but is far less damaging. This asymmetry makes **recall** the most appropriate evaluation metric for this class.

---

### Threshold Selected

A threshold of **0.10** was chosen after scanning thresholds from 0.05 to 0.30. At 0.10, approximately 8% of test samples are flagged as High (vs. the true 3.3%), yielding the largest recall gain before precision degrades to a point where the flag becomes operationally meaningless. Thresholds below 0.10 flagged over 11% of samples — more than three times the true prevalence — with precision falling to 0.27. Higher thresholds (0.20–0.30) preserved more precision but gave up substantial recall, which is the metric we are optimizing.

---

### Metric Change and Tradeoff

| | High Recall | High Precision | High F1 | Overall Accuracy |
|---|---|---|---|---|
| Baseline (default rule) | 0.42 | 0.72 | 0.53 | 83% |
| Threshold = 0.10 | **0.87** | 0.36 | 0.51 | 80% |

**Tradeoff observed:** Lowering the threshold to 0.10 more than doubled High recall (0.42 → 0.87) — the model now catches 87% of true high-need fields instead of only 42%. The cost is steep: precision fell from 0.72 to 0.36, meaning roughly two out of every three fields flagged as High are actually Low or Medium. Overall accuracy dropped 3 points (83% → 80%), and High F1 declined slightly (0.53 → 0.51) because the precision loss roughly offset the recall gain. This illustrates the core tradeoff clearly: the threshold is a lever that trades false negatives for false positives without changing the underlying model. Whether 0.10 is the right operating point depends on how much unnecessary irrigation (false positives) is acceptable to avoid missing a high-need field (false negative).

---

### Gaussian Naive Bayes vs. Existing Models

GaussianNB performs substantially worse than the tree-based models trained for this competition:

| Model | CV / Holdout Accuracy | Notes |
|---|---|---|
| Random Forest (tuned) | 98.55% | 3-fold CV, full training data |
| XGBoost | ~98.5% | 3-fold CV, full training data |
| LightGBM / CatBoost | ~98.5% | Similar to above |
| **Gaussian Naive Bayes** | **83%** | Holdout (20%), full training data |

GNB achieves 83% holdout accuracy — about 15–16 percentage points below the tree-based ensemble methods. The gap is driven by three structural limitations:

1. **Feature independence assumption.** GNB treats all features as conditionally independent given the class. This dataset has strong feature interactions — `Crop_Growth_Stage` and `Soil_Moisture` together account for ~58% of RF feature importance. GNB cannot capture these joint effects; it multiplies independent per-feature likelihoods instead, which underestimates the actual discriminative signal.

2. **Distributional mismatch.** GNB assumes each feature follows a Gaussian distribution within each class. The eight categorical features encoded as ordinal integers are clearly not Gaussian, and several numerical features are skewed or bounded. `StandardScaler` normalizes scale but cannot fix distributional shape — the model's likelihood estimates for these features are systematically off.

3. **No non-linear boundaries.** Tree models recursively split on feature combinations and can approximate arbitrarily complex decision boundaries. GNB applies a fixed Gaussian likelihood model and cannot learn that, for example, high soil moisture matters much more when the crop is at a late growth stage.

Despite its lower accuracy, GNB is essentially instant to train on 630K rows and naturally produces calibrated probability estimates, making it a useful fast baseline that cleanly quantifies the cost of ignoring feature dependence and distributional assumptions.